# 02 - Descriptor stability (MANDATORY before Phase 1)

AGENTS.md Sec 3.3: if the per-class image quota is too small, the class mean and within-class
covariance are noisy, and that INPUT noise depresses Phase-1 R2. A negative Phase-1 result would
then be **ambiguous** - no signal, or a bad descriptor? This run removes the ambiguity.

## Why the per-stratum part is not optional on Pl@ntNet

Extraction with quota=100 returned only **45,756** images, not 100 x 1081 = 108,100 - so the
average class supplies ~42 train images and the descriptor side is **severely long-tailed**.
If descriptors are noisier for rare classes then **descriptor quality correlates with prevalence**,
and Sec 6.3 gate C asks precisely whether geometry beats log-prevalence. Geometry could then look
predictive exactly where prevalence is high - a spurious gate-C pass.

So this notebook reports **two things**, and the second one gates the interpretation of gate C:
1. the overall stability curve (which quota is enough),
2. **stability per prevalence quartile**, and the head-minus-tail spread.

## Estimator: disjoint halves vs bootstrap

Disjoint halves need `2q` images per class, which the rare strata simply do not have (measured on
a Pl@ntNet-like simulation: the two rarest quartiles were unmeasurable even at q=5). A **bootstrap**
pair (two independent resamples of q images, with replacement) works whenever `q <= n_y`.

**The two are NOT comparable to each other** - bootstrap resamples share samples so its
correlations are biased upward. They ARE comparable across strata at fixed method and q, which is
all the head/tail question needs. The `method` field is recorded in every result.


## 1. Config - `# === EDIT ME ===`


In [ ]:
# === EDIT ME ===========================================================
REPO_URL   = ''
REPO_DIR   = 'foundation-cp'
DRIVE_ROOT = '/content/drive/MyDrive/pcc'

DATASET    = 'plantnet'      # 'plantnet' | 'cifar100'
BACKBONE   = 'resnet50_ltc'  # 'resnet50_ltc' | 'resnet50_self'
SPLIT      = 'train_quota'   # descriptors come from TRAINING data only (Sec 6.3)

QUOTAS     = (5, 10, 25, 50)   # disjoint halves need 2q per class; bootstrap needs q
N_REPS     = 3
N_STRATA   = 4
STABLE_THRESHOLD = 0.90
SEED = 42
EMB_ROOT = f'{DRIVE_ROOT}/embeddings/{DATASET}/{BACKBONE}'
# =======================================================================
print('EMB_ROOT =', EMB_ROOT, '| split', SPLIT, '| quotas', QUOTAS)


## 2. Mount Drive + repo + env


In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
if REPO_URL and not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone',REPO_URL,REPO_DIR], check=True)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR if os.path.isabs(REPO_DIR) else '/content/'+REPO_DIR)
os.environ['PYTHONPATH'] = os.getcwd() + os.pathsep + os.environ.get('PYTHONPATH','')
subprocess.run(['pip','install','-q','-r','requirements.txt'], check=False)
from pcc.utils.seed import set_seed; from pcc.utils.io import environment_stamp
set_seed(SEED)
print('env:', environment_stamp()['packages'])


## 3. Load the descriptor source (verified shards) + provenance


In [ ]:
import numpy as np
from pcc.data.load import load_split, split_provenance, per_class_counts
from pcc.data.ltc_datasets import NUM_CLASSES

prov = split_provenance(EMB_ROOT, SPLIT)
if prov.get('under_gate_exception'):
    print('NOTE: data extracted under a GATE EXCEPTION -', prov.get('scores_source'))
d = load_split(EMB_ROOT, SPLIT)
emb, lab = np.asarray(d['embeddings']), np.asarray(d['labels']).astype(int)
lg = np.asarray(d['logits'])
K = NUM_CLASSES[DATASET] if DATASET in NUM_CLASSES else int(lab.max()) + 1
counts = per_class_counts(lab, K)
nz = counts[counts > 0]
print(f'embeddings {emb.shape} | classes with images: {(counts>0).sum()}/{K}')
print('images/class percentiles:',
      {f'p{q}': int(np.percentile(nz, q)) for q in (0,10,25,50,75,90,100)})
print(f'classes with >=2q images for q={max(QUOTAS)}: {(counts >= 2*max(QUOTAS)).sum()}')


## 4. Overall stability curve (bootstrap, so every class can contribute)


In [ ]:
from pcc.descriptors.stability import descriptor_stability

res = descriptor_stability(emb, lg, lab, K, quotas=QUOTAS, n_reps=N_REPS,
                           seed=SEED, stable_threshold=STABLE_THRESHOLD,
                           method='bootstrap')
print(f"{'quota':>6} {'mean_corr':>10} {'se':>8} {'classes':>8}")
for q in QUOTAS:
    v = res['by_quota'].get(q, {})
    if v.get('insufficient_data'):
        print(f'{q:>6}   insufficient'); continue
    print(f"{q:>6} {v['mean_corr']:>10.3f} {v['se']:>8.3f} {v['n_classes_used']:>8}")
print('\nrecommended_quota:', res['recommended_quota'],
      f"(threshold {STABLE_THRESHOLD}, method {res['method']})")


## 5. Per-feature detail - which descriptors are the noisy ones


In [ ]:
names = res['feature_names']
avail = [q for q in QUOTAS if not res['by_quota'].get(q,{}).get('insufficient_data')]
print('feature'.ljust(20) + ''.join(f'q={q}'.rjust(9) for q in avail))
for nm in names:
    row = ''.join(f"{res['by_quota'][q]['per_feature'][nm]:9.3f}" for q in avail)
    print(nm.ljust(20) + row)
print('\nexcluded from the aggregate (constant by construction):',
      res['by_quota'][avail[0]]['excluded_from_aggregate'])


## 6. GATE-C CONTAMINATION GUARD - stability per prevalence quartile

A large head-minus-tail spread means descriptor quality tracks prevalence. Any gate-C conclusion
MUST be reported next to this number, because it is a live alternative explanation for geometry
looking predictive.


In [ ]:
from pcc.descriptors.stability import descriptor_stability_by_stratum

strat = descriptor_stability_by_stratum(emb, lg, lab, K, counts, quotas=QUOTAS,
                                        n_reps=N_REPS, seed=SEED,
                                        stable_threshold=STABLE_THRESHOLD,
                                        n_strata=N_STRATA, method='bootstrap')
print('stratum'.ljust(24) + ''.join(f'q={q}'.rjust(9) for q in QUOTAS) + '   classes')
for k, v in strat['by_stratum'].items():
    if v.get('insufficient_data'):
        print(k.ljust(24) + '  INSUFFICIENT'); continue
    row = ''.join((f"{v['by_quota'][q]['mean_corr']:9.3f}"
                   if not v['by_quota'][q].get('insufficient_data') else '      n/a')
                  for q in QUOTAS)
    print(k.ljust(24) + row + f"   {v['n_classes_in_stratum']:5d}")
print()
print('HEAD - TAIL spread (the contamination warning):')
for q, sp in strat['spread'].items():
    hmt = sp.get('head_minus_tail')
    print(f"  q={q:3d}: tail={sp['tail']} head={sp['head']} "
          f"head-tail={'n/a' if hmt is None else round(hmt,3)}")
print()
print('unevaluable classes (no images at all):', strat['unevaluable_classes'])


## 7. Write report


In [ ]:
import time
from pcc.utils.io import write_report

def clean(o):
    if isinstance(o, dict): return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [clean(v) for v in o]
    if isinstance(o, np.ndarray): return None
    if isinstance(o, (np.floating, np.integer)): return float(o)
    return o

rec = res['recommended_quota']
spread_max = max([abs(v['head_minus_tail']) for v in strat['spread'].values()
                  if v.get('head_minus_tail') is not None] or [float('nan')])
conclusion = (f'recommended_quota={rec}' if rec is not None else
              f'NO quota reached {STABLE_THRESHOLD} - Phase-1 negatives would be ambiguous')
conclusion += f' | max |head-tail| spread={spread_max:.3f}'

report = write_report('pcc/reports', f'02_descriptor_stability_{DATASET}',
    hypothesis='class descriptors phi(y) stabilize as images-per-class grows, and descriptor '
               'quality does NOT depend strongly on class prevalence',
    pass_criteria=f'report the stability curve with SE for q in {QUOTAS}; recommended_quota = '
                  f'smallest q with mean cross-draw correlation >= {STABLE_THRESHOLD}; ALSO '
                  f'report stability per prevalence quartile and the head-minus-tail spread, '
                  f'which must accompany any gate-C conclusion. If no q reaches the threshold, '
                  f'report that as the finding rather than taking the largest q.',
    config=dict(dataset=DATASET, backbone=BACKBONE, split=SPLIT, quotas=list(QUOTAS),
                n_reps=N_REPS, n_strata=N_STRATA, method='bootstrap',
                stable_threshold=STABLE_THRESHOLD,
                under_gate_exception=bool(prov.get('under_gate_exception'))),
    seed=SEED,
    results={'overall': clean(res), 'by_stratum': clean(strat),
             'images_per_class_percentiles': {f'p{q}': int(np.percentile(nz, q))
                                             for q in (0,10,25,50,75,90,100)}},
    conclusion=conclusion, started_at=time.time())
print('report:', report)
print()
print(conclusion)
if rec is not None:
    mark = f'{DRIVE_ROOT}/gates/DESCRIPTOR_QUOTA_{DATASET}.json'
    os.makedirs(os.path.dirname(mark), exist_ok=True)
    import json as _j
    _j.dump({'dataset':DATASET,'recommended_quota':rec,'threshold':STABLE_THRESHOLD,
             'method':'bootstrap','head_tail_spread':float(spread_max)}, open(mark,'w'), indent=2)
    print('quota marker ->', mark)
else:
    print('NO marker written - decide with a human before running Phase 1 (Sec 3.3).')
